# Chapter 4 - Practice 3: Get started with Hugging Face

---  
## Exercise 1: Sentiment Analysis with Hugging Face

**Yêu cầu:**
1. Sử dụng mô hình phân tích cảm xúc pre-trained từ Hugging Face Hub.
2. Tokenize câu văn mẫu.
3. Thực hiện phân tích cảm xúc trên câu văn đó.

In [2]:
# Import các thư viện
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Bước 2: Tải mô hình Pre-trained và Tokenizer
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
print(f"Đang tải mô hình pre-trained: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

Đang tải mô hình pre-trained: distilbert-base-uncased-finetuned-sst-2-english


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 493.76it/s]


In [3]:
# Bước 3 & 4: Tokenize và Dự đoán Cảm xúc (Explicit steps)
sample_sentences = [
    "I absolutely love learning deep learning with PyTorch and Hugging Face!",
    "The weather today is terrible and I feel sick.",
    "The course material is okay, but could be improved."
]

print("--- KẾT QUẢ DỰ ĐOÁN TỪNG CÂU ---")
for sentence in sample_sentences:
    # Tokenize câu văn
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
    
    # Forward pass qua mô hình
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        pred_class = torch.argmax(probs, dim=-1).item()
        label = model.config.id2label[pred_class]
        confidence = probs[0][pred_class].item()
        
    print(f"\nCâu: '{sentence}'")
    print(f"Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
    print(f"-> Dự đoán: {label} (Độ tin cậy: {confidence:.4f})")

--- KẾT QUẢ DỰ ĐOÁN TỪNG CÂU ---

Câu: 'I absolutely love learning deep learning with PyTorch and Hugging Face!'
Tokens: ['[CLS]', 'i', 'absolutely', 'love', 'learning', 'deep', 'learning', 'with', 'p', '##yt', '##or', '##ch', 'and', 'hugging', 'face', '!', '[SEP]']
-> Dự đoán: POSITIVE (Độ tin cậy: 0.9998)

Câu: 'The weather today is terrible and I feel sick.'
Tokens: ['[CLS]', 'the', 'weather', 'today', 'is', 'terrible', 'and', 'i', 'feel', 'sick', '.', '[SEP]']
-> Dự đoán: NEGATIVE (Độ tin cậy: 0.9995)

Câu: 'The course material is okay, but could be improved.'
Tokens: ['[CLS]', 'the', 'course', 'material', 'is', 'okay', ',', 'but', 'could', 'be', 'improved', '.', '[SEP]']
-> Dự đoán: POSITIVE (Độ tin cậy: 0.8978)


In [4]:
# Sử dụng Pipeline API rút gọn của Hugging Face
classifier = pipeline("sentiment-analysis", model=model_name)
results = classifier(sample_sentences)

print("--- KẾT QUẢ SỬ DỤNG PIPELINE API ---")
for sent, res in zip(sample_sentences, results):
    print(f"Câu: '{sent}' -> {res['label']} ({res['score']:.4f})")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 157.21it/s]


--- KẾT QUẢ SỬ DỤNG PIPELINE API ---
Câu: 'I absolutely love learning deep learning with PyTorch and Hugging Face!' -> POSITIVE (0.9998)
Câu: 'The weather today is terrible and I feel sick.' -> NEGATIVE (0.9995)
Câu: 'The course material is okay, but could be improved.' -> POSITIVE (0.8978)


---  
## Exercise 2: Finetuning a Pretrained Model for Binary Text Classification

**Yêu cầu:**
1. Cài đặt các thư viện Hugging Face (`transformers`, `datasets`, `evaluate`).
2. Tải tập dữ liệu phân loại văn bản nhị phân từ Kaggle hoặc file CSV local.
3. Tải mô hình Pre-trained và Tokenizer tương ứng.
4. Tiền xử lý (tokenization) tập dữ liệu.
5. Định nghĩa các tham số huấn luyện (`TrainingArguments`).
6. Khởi tạo đối tượng `Trainer` và tiến hành Fine-tune mô hình.
7. Đánh giá mô hình sau khi Fine-tune.

In [5]:
import os
import pandas as pd
import numpy as np
import evaluate
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [6]:
# Bước 2: Tải tập dữ liệu nhị phân (Hỗ trợ nạp file 'IMDB Dataset.csv' tải từ Kaggle)
# Link Kaggle Dataset: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

kaggle_file = "IMDB Dataset.csv"
local_train = "train_data.csv"

if os.path.exists(kaggle_file):
    print(f"Đã tìm thấy dataset Kaggle: '{kaggle_file}'!")
    df = pd.read_csv(kaggle_file)
    # Tiền xử lý dữ liệu Kaggle IMDB Dataset.csv (chuyển sentiment -> label 0/1)
    df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})
    df = df.rename(columns={'review': 'text'})
    full_ds = Dataset.from_pandas(df[['text', 'label']])
    # Lấy mẫu 500 train và 100 eval để demo huấn luyện nhanh
    split_ds = full_ds.train_test_split(test_size=0.2, seed=42)
    train_dataset = split_ds['train'].select(range(min(500, len(split_ds['train']))))
    eval_dataset = split_ds['test'].select(range(min(100, len(split_ds['test']))))
elif os.path.exists(local_train):
    print(f"Sử dụng tập dữ liệu CSVlocal: '{local_train}'")
    dataset = load_dataset("csv", data_files={"train": "train_data.csv", "test": "test_data.csv"})
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]
else:
    print("Tải dataset online SST2:")
    raw_datasets = load_dataset("sst2")
    train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(500))
    eval_dataset = raw_datasets["validation"].shuffle(seed=42).select(range(100))

print(f"Kích thước tập Train: {len(train_dataset)}")
print(f"Kích thước tập Test/Eval: {len(eval_dataset)}")
print("Mẫu dữ liệu đầu tiên:", train_dataset[0])

Sử dụng tập dữ liệu CSVlocal: 'train_data.csv'


Generating train split: 500 examples [00:00, 13620.44 examples/s]
Generating test split: 100 examples [00:00, 27666.91 examples/s]

Kích thước tập Train: 500
Kích thước tập Test/Eval: 100
Mẫu dữ liệu đầu tiên: {'text': 'I really enjoyed watching this film. Fantastic cinematography and brilliant direction.', 'label': 1}


In [7]:
# Bước 3: Tải mô hình Pre-trained và Tokenizer
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 405.52it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# Bước 4: Tiền xử lý dữ liệu (Tokenization)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map: 100%|██████████| 100/100 [00:00<00:00, 3563.22 examples/s]


In [9]:
# Hàm tính toán độ chính xác (Accuracy)
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [10]:
# Bước 5: Định nghĩa Training Arguments
training_args = TrainingArguments(
    output_dir="./results_exercise2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

In [11]:
# Bước 6: Tạo Trainer và tiến hành Fine-tune mô hình
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Đang tiến hành huấn luyện (Fine-tuning)...")
trainer.train()

Đang tiến hành huấn luyện (Fine-tuning)...


C:\Users\caibo\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.020471,0.013186,1.000000
2,0.008884,0.006870,1.000000


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.34s/it]
C:\Users\caibo\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.32s/it]


TrainOutput(global_step=126, training_loss=0.123043877116981, metrics={'train_runtime': 98.0195, 'train_samples_per_second': 10.202, 'train_steps_per_second': 1.285, 'total_flos': 4839199657152.0, 'train_loss': 0.123043877116981, 'epoch': 2.0})

In [12]:
# Bước 7: Đánh giá mô hình sau khi Fine-tune
eval_results = trainer.evaluate()
print("\n--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---")
print(eval_results)

C:\Users\caibo\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.008884,0.013186,2,1.000000



--- KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ---
{'eval_loss': 0.01318590622395277, 'eval_accuracy': 1.0}
